# Maneuvering derivatives from a surface boundary layer and shed vorticity

This notebook is the end of the chain that notebooks 01 to 04 begin. It shows the
whole model working on KVLCC2, and — more usefully — where each piece of the
physics enters and how much of the answer it is responsible for.

The short version of the story. Ideal flow gives a ship at drift **no side force
at all**: d'Alembert's paradox, and the boundary-element solve returns exactly
that. Every bit of $Y_v$ therefore comes from the flow being viscous, and it
arrives through two quite different routes:

1. the **displacement effect** of the attached boundary layer, which perturbs the
   outer flow and so the pressure, and
2. the **vorticity shed** where the flow leaves the hull, which carries lateral
   momentum away instead of returning it.

The second is much the larger. Notebooks 02 and 04 had only the first, which is
why they reached about a sixth of the measured $Y_v$.

## The physics, and what is modelled

| | Physics | How it is treated here |
|---|---|---|
| 1 | Inviscid outer flow (double body) | Exact, boundary elements. Gives $Y_v=0$ and the Munk moment |
| 2 | Attached boundary layer: friction and displacement | Three-dimensional integral boundary layer on the surface |
| 3 | Three-dimensional separation | Domain terminated where the thin-layer assumption fails |
| 4 | Bilge vortices, linear part | Slender-body momentum balance terminated at separation |
| 5 | Bilge vortices, quadratic part | Sectional cross-flow drag |
| 6 | Free surface and wave making | Excluded — this is a double body at low Froude number |

Rows 4 and 5 are *superposed* on row 2 rather than embedded in it. That follows
Tanaka, who argues the ordinary boundary layer and the longitudinal separating
vortices "orthogonally intersect, so the characteristics of each vortex are
probably approximated to be independent of each other". The coupling is one way:
the boundary layer establishes where the flow leaves the hull, and the vortex
model takes the momentum from there.

In [1]:
using LinearAlgebra
using MarineHydro
using Printf
using TOML

data_directory = joinpath("..", "validation", "gothenburg2010", "data", "KVLCC2")
surface_paths = [joinpath(data_directory, "kvlcc_bow1.dat"),
    joinpath(data_directory, "kvlcc2_stn1.dat")]
all(isfile, surface_paths) ||
    error("KVLCC2 geometry missing; run validation/gothenburg2010/fetch_geometry.sh")

REYNOLDS_NUMBER = 4.6e6
FROUDE_NUMBER = 0.142
WATER_DENSITY = 1025.0
forward_speed = FROUDE_NUMBER * sqrt(SETTINGS.g)
kinematic_viscosity = forward_speed / REYNOLDS_NUMBER
ittc_1957 = 0.075 / (log10(REYNOLDS_NUMBER) - 2)^2

grid = read_gothenburg2010_panel_grid(surface_paths; target_shape = (16, 9))
mesh = grid.mesh
ship_length = maximum(mesh.centers[:, 1]) - minimum(mesh.centers[:, 1])
draft = -minimum(mesh.centers[:, 3])
midship = (maximum(mesh.centers[:, 1]) + minimum(mesh.centers[:, 1])) / 2
(; panels = mesh.nfaces, ship_length, draft, forward_speed)

(panels = 480, ship_length = 1.044009937059943, draft = 0.06500000000000002, forward_speed = 0.4447570572795894)

## The measurement to beat

MMG normalises on $\tfrac12\rho L T U^2$ where the formulation here uses
$\tfrac12\rho L^2U^2$, so the published set is converted with
$\lambda_T=T/L$ before comparison. Note the sign convention: $Y_v<0$ means a
drift to starboard is resisted.

In [2]:
reference = TOML.parsefile(joinpath("..", "validation", "kvlcc2_maneuvering",
    "reference.toml"))
mmg = reference["mmg"]["linear_hull"]
target = mmg_to_wang_velocity_derivatives(
    MMGLinearHullDerivatives(mmg["Y_v"], mmg["Y_r"], mmg["N_v"], mmg["N_r"]),
    draft, ship_length)
(; Y_v = target.Y_v, N_v = target.N_v, note = "Wang normalisation")

(Y_v = -0.01961188229458817, N_v = -0.008529612299551046, note = "Wang normalisation")

## Step 1 — the inviscid solve, and the paradox

The double-body potential flow is solved once and supplies both the rigid-body
potential gradients the boundary layer needs for its edge velocity, and the
ideal-flow derivatives. The side force it predicts is zero to solver accuracy.
That is not a defect: it is d'Alembert, and it is why a purely inviscid
maneuvering prediction has nothing to say about $Y_v$.

In [3]:
surge = solve_rigid_body_potential(mesh, :surge)
sway = solve_rigid_body_potential(mesh, :sway)
yaw = solve_rigid_body_potential(mesh, :yaw)

edge_velocity = body_relative_edge_velocity(mesh, forward_speed, 0.0, 0.0;
    surge_gradient = surge.potential_gradient,
    sway_gradient = sway.potential_gradient,
    yaw_gradient = yaw.potential_gradient)
@printf("edge speed: min %.4f  max %.4f  (freestream %.4f)\n",
    minimum(norm, eachrow(edge_velocity)), maximum(norm, eachrow(edge_velocity)),
    forward_speed)

edge speed: min 0.1469  max 0.5134  (freestream 0.4448)


## Step 2 — where the boundary layer is allowed to apply

The integral equations assume the layer is thin against the local radius of
curvature. Measuring $\delta\kappa=\delta/R$ along the hull shows that at the
stern this is not merely marginal but inverted — the layer is several times
*thicker* than the local radius. Tanaka names this directly: where the radius is
small "the basic governing equations become different from the ordinary ones",
and first-order integral methods "are not applicable" near the stern end of a
full-form ship.

So the domain is terminated where $\delta\kappa$ exceeds a limit. This is a
statement about the validity of the equations, not a numerical convenience, and
it replaces the fitted Schmitz stern cut used in notebook 02.

In [4]:
topology = build_surface_topology(mesh)
metrics = build_surface_metrics(mesh, topology)

curvature = [MarineHydro.surface_curvature(mesh, topology, panel)
             for panel in 1:mesh.nfaces]
stern = minimum(mesh.centers[:, 1])
println("delta*kappa by tenth of the length from the stern")
println(rpad("x/L", 12), rpad("panels", 9), rpad("mean R", 12), "mean curvature")
for band in 0:9
    inside = findall(panel -> band / 10 <=
                              (mesh.centers[panel, 1] - stern) / ship_length <
                              (band + 1) / 10 + 1e-9, 1:mesh.nfaces)
    isempty(inside) && continue
    mean_curvature = sum(curvature[inside]) / length(inside)
    @printf("%.1f-%.1f     %-9d%-12.4g%.4g\n", band / 10, (band + 1) / 10,
        length(inside), 1 / max(mean_curvature, 1e-9), mean_curvature)
end

delta*kappa by tenth of the length from the stern
x/L         panels   mean R      mean curvature
0.0-0.1     114      0.007828    127.7


0.1-0.2     54       0.04148     24.11
0.2-0.3     36       0.06753     14.81
0.3-0.4     20       0.06763     14.79
0.4-0.5     16       0.05371     18.62
0.5-0.6     16       0.05265     18.99
0.6-0.7     16       0.05116     19.55
0.7-0.8     32       0.0568      17.61
0.8-0.9     44       0.06995     14.3
0.9-1.0     132      0.01623     61.61


In [5]:
active, retained = attached_flow_domain(mesh, edge_velocity, kinematic_viscosity;
    topology, metrics, curvature_limit = 0.5, minimum_retained = 0.0)
@printf("retained %.1f%% of the wetted area, %d of %d panels\n",
    100 * retained, count(active), mesh.nfaces)

retained 84.5% of the wetted area, 380 of 480 panels


## Step 3 — the surface boundary layer

The layer is solved as one globally coupled system over the retained domain, not
marched along mesh lines. Two momentum-defect equations and a kinetic-energy
equation, cell-centred finite volume, upwinded, in a local Cartesian basis per
panel — the formulation of Lokatt and Eller. The wall-normal direction has been
integrated out, so the computational domain is the hull surface itself: a
two-parametric surface in three space dimensions. "Three-dimensional" refers to
the physics, the crossflow, not to the domain.

Read the reported residual honestly. The solve does not reach its tolerance on
this hull; the friction coefficient it produces is nonetheless stable across mesh
refinement and consistent with the correlation, which is what the loads depend
on.

In [6]:
layer = solve_surface_boundary_layer(mesh, edge_velocity, kinematic_viscosity;
    rho = WATER_DENSITY, active, topology, metrics)
retained_area = sum(mesh.areas[active])
friction = -layer.force[1] / (0.5 * WATER_DENSITY * forward_speed^2 * retained_area)
@printf("C_F = %.5g   = %.3f x ITTC-1957\n", friction, friction / ittc_1957)
@printf("mean |crossflow| = %.3g deg   max = %.3g deg\n",
    rad2deg(sum(abs, layer.crossflow_angle[active]) / count(active)),
    rad2deg(maximum(abs, layer.crossflow_angle[active])))
@printf("solver: residual %.4g   converged = %s   <- not converged, see above\n",
    layer.diagnostics.residual_norm, layer.diagnostics.converged)

C_F = 0.0031799   = 0.922 x ITTC-1957


mean |crossflow| = 6.55 deg   max = 52.4 deg


solver: residual 6.663   converged = false   <- not converged, see above


## Step 4 — the displacement route to $Y_v$

The layer's displacement thickness is imposed on the inviscid problem as a
transpiration velocity; the resulting potential is differentiated with respect
to sway and yaw and integrated for the pressure contribution, and the wall shear
adds a direct contribution. This is the whole of what notebooks 02 and 04 had.

It is small — of order a sixth of the measurement — and that shortfall is the
entire motivation for what follows.

In [7]:
correction = viscous_maneuvering_correction(grid, surge.potential_gradient,
    sway.potential_gradient, yaw.potential_gradient, forward_speed,
    kinematic_viscosity; rho = WATER_DENSITY, linearization = :central)
viscous = nondimensionalize_viscous_derivatives(correction.derivatives, ship_length,
    forward_speed; rho = WATER_DENSITY)
@printf("displacement + friction:  Y_v' = %+.5g   N_v' = %+.5g\n",
    viscous.Y_v, viscous.N_v)
@printf("measured (MMG):           Y_v' = %+.5g   N_v' = %+.5g\n",
    target.Y_v, target.N_v)
@printf("fraction of measured Y_v: %.2f\n", viscous.Y_v / target.Y_v)

displacement + friction:  Y_v' = -0.0039883   N_v' = +0.0015821
measured (MMG):           Y_v' = -0.019612   N_v' = -0.0085296
fraction of measured Y_v: 0.20


## Step 5 — the shed vorticity

A fluid section sweeping aft carries lateral momentum $m_{22}(x)\,v$, so in
steady flow the force on the hull is the rate at which that momentum changes,

$$f_y(x)=U\,v\,\frac{\mathrm{d}m_{22}}{\mathrm{d}x}.$$

On a body that recovers its momentum the integral telescopes to zero — that is
d'Alembert again, from the slender-body side. Where the flow leaves the surface
at $x_s$ the momentum held there is carried into the trailing vortices instead:

$$Y_v=-U\,m_{22}(x_s),\qquad
N_v=-U\left[(x_s-x_0)m_{22}(x_s)+\int_{x_s}^{\text{bow}}m_{22}\,\mathrm{d}x\right].$$

Two things are worth stating carefully. The balance telescopes to zero only when
$m_{22}$ vanishes at the tail, and $m_{22}$ scales as the square of the **draft**,
not the beam — a hull carrying full draft to its stern does not satisfy
d'Alembert here. And consequently $Y_v\simeq-2\pi(T/L)^2$, largely *insensitive*
to exactly where the separation station falls. The boundary layer's job is to
establish that the flow leaves the after body, not to locate it to the panel.

In [8]:
sections = sectional_crossflow_geometry(grid, falses(mesh.nfaces))
excluded = findall(.!active)
separation_x = isempty(excluded) ? stern : maximum(mesh.centers[excluded, 1])

shed = shed_vorticity_derivatives(sections, separation_x, forward_speed;
    rho = WATER_DENSITY, x_reference = midship)
force_scale = WATER_DENSITY / 2 * ship_length^2 * forward_speed
moment_scale = WATER_DENSITY / 2 * ship_length^3 * forward_speed
@printf("separation station x/L = %.3f from the stern\n",
    (separation_x - stern) / ship_length)
@printf("shed vorticity:  Y_v' = %+.5g   N_v' = %+.5g\n",
    shed.Y_v / force_scale, shed.N_v / moment_scale)
@printf("slender-body estimate -2*pi*(T/L)^2 = %+.5g\n",
    -2π * (draft / ship_length)^2)

separation station x/L = 0.642 from the stern
shed vorticity:  Y_v' = -0.024356   N_v' = -0.010798
slender-body estimate -2*pi*(T/L)^2 = -0.024356


## The assembled answer

In [9]:
shed_y = shed.Y_v / force_scale
shed_n = shed.N_v / moment_scale
println(rpad("contribution", 28), rpad("Y_v'", 14), "N_v'")
@printf("%-28s%-14.5g%.5g\n", "ideal flow", 0.0, 0.0)
@printf("%-28s%-14.5g%.5g\n", "displacement + friction", viscous.Y_v, viscous.N_v)
@printf("%-28s%-14.5g%.5g\n", "shed vorticity", shed_y, shed_n)
println(repeat("-", 56))
@printf("%-28s%-14.5g%.5g\n", "total", viscous.Y_v + shed_y, viscous.N_v + shed_n)
@printf("%-28s%-14.5g%.5g\n", "measured (MMG)", target.Y_v, target.N_v)
@printf("%-28s%-14.2f%.2f\n", "ratio to measured",
    (viscous.Y_v + shed_y) / target.Y_v, (viscous.N_v + shed_n) / target.N_v)

contribution                Y_v'          N_v'
ideal flow                  0             0


displacement + friction     -0.0039883    0.0015821
shed vorticity              -0.024356     -0.010798
--------------------------------------------------------
total                       -0.028344     -0.0092158
measured (MMG)              -0.019612     -0.0085296


ratio to measured           1.45          1.08


The total over-predicts $Y_v$ by roughly a half. That is the expected
direction and worth naming rather than tuning away: the momentum balance assumes
**all** the lateral momentum at the separation station is lost, which is an upper
bound, and the sectional added mass uses the flat-plate coefficient of one rather
than a Lewis-form factor for a full section. Both push the same way. Nothing here
is fitted to the measurement.

## Mesh convergence

The interesting quantity is the shed term, since it is the large one. It should
be insensitive to the mesh, and to the separation station, for the reason given
above — and it is, to about a per cent over a fourfold refinement.

In [10]:
println(rpad("panels", 9), rpad("x_s/L", 9), rpad("Y_v' shed", 13),
    rpad("Y_v' visc", 13), rpad("Y_v' total", 13), "C_F/ITTC")
for shape in ((12, 7), (16, 9), (20, 11), (24, 13))
    local g = read_gothenburg2010_panel_grid(surface_paths; target_shape = shape)
    local m = g.mesh
    local L = maximum(m.centers[:, 1]) - minimum(m.centers[:, 1])
    local mid = (maximum(m.centers[:, 1]) + minimum(m.centers[:, 1])) / 2
    local sg = solve_rigid_body_potential(m, :surge)
    local sw = solve_rigid_body_potential(m, :sway)
    local yw = solve_rigid_body_potential(m, :yaw)
    local ev = body_relative_edge_velocity(m, forward_speed, 0.0, 0.0;
        surge_gradient = sg.potential_gradient, sway_gradient = sw.potential_gradient,
        yaw_gradient = yw.potential_gradient)
    local tp = build_surface_topology(m)
    local mt = build_surface_metrics(m, tp)
    local act, _ = attached_flow_domain(m, ev, kinematic_viscosity; topology = tp,
        metrics = mt, curvature_limit = 0.5, minimum_retained = 0.0)
    local lay = solve_surface_boundary_layer(m, ev, kinematic_viscosity;
        rho = WATER_DENSITY, active = act, topology = tp, metrics = mt)
    local sec = sectional_crossflow_geometry(g, falses(m.nfaces))
    local exc = findall(.!act)
    local xs = isempty(exc) ? minimum(m.centers[:, 1]) : maximum(m.centers[exc, 1])
    local sh = shed_vorticity_derivatives(sec, xs, forward_speed; rho = WATER_DENSITY,
        x_reference = mid)
    local corr = viscous_maneuvering_correction(g, sg.potential_gradient,
        sw.potential_gradient, yw.potential_gradient, forward_speed,
        kinematic_viscosity; rho = WATER_DENSITY, linearization = :central)
    local vis = nondimensionalize_viscous_derivatives(corr.derivatives, L,
        forward_speed; rho = WATER_DENSITY)
    local shedy = sh.Y_v / (WATER_DENSITY / 2 * L^2 * forward_speed)
    local cf = -lay.force[1] /
               (0.5 * WATER_DENSITY * forward_speed^2 * sum(m.areas[act]))
    @printf("%-9d%-9.3f%-13.5g%-13.5g%-13.5g%.4f\n", m.nfaces,
        (xs - minimum(m.centers[:, 1])) / L, shedy, vis.Y_v, shedy + vis.Y_v,
        cf / ittc_1957)
end

panels   x_s/L    Y_v' shed    Y_v' visc    Y_v' total   C_F/ITTC
264      0.152    -0.024108    -0.0023303   -0.026439    0.9162


480      0.642    -0.024356    -0.0039883   -0.028344    0.9218


760      0.673    -0.024299    -0.004481    -0.02878     0.9579


1104     0.701    -0.024246    -0.0042479   -0.028493    0.9451


## The nonlinear half, and a virtual captive test

Cross-flow drag is quadratic in the drift, so it contributes nothing to the
linear derivatives above — the tests assert exactly that. It shows up instead in
the cubic coefficient, which a linearisation about straight-ahead cannot reach at
all. The virtual captive test sweeps drift angle, tares on the straight-ahead
run as a towing tank tares its balance, and regresses the coefficients.

In [11]:
captive = virtual_captive_test(grid, surge.potential_gradient,
    sway.potential_gradient, yaw.potential_gradient, forward_speed,
    kinematic_viscosity; drift_angles = range(-0.14, 0.14; length = 9),
    yaw_rates = [0.0], rho = WATER_DENSITY, x_reference = midship)
@printf("Y_v'   = %+.5g      Y_vvv' = %+.5g\n", captive.coefficients.sway.Y_v,
    captive.coefficients.sway.Y_vvv)
@printf("N_v'   = %+.5g      N_vvv' = %+.5g\n", captive.coefficients.yaw.N_v,
    captive.coefficients.yaw.N_vvv)
@printf("fit residual %.2e; %d of %d runs converged\n",
    captive.coefficients.residual.sway, count(run -> run.converged, captive.runs),
    length(captive.runs))

Y_v'   = -0.0016954      Y_vvv' = +0.020773
N_v'   = +0.00048995      N_vvv' = -0.0096277


fit residual 2.21e-02; 9 of 9 runs converged


The cubic term dominates over this range: the response at eight degrees of
drift is far from what the linear derivative alone predicts, which is
information the linearisation structurally cannot produce.

## Sensitivity with respect to the mesh

The derivative of a hull integral with respect to a shape parameter comes
through `deform_mesh`, which displaces an imported hull's vertices and recomputes
the panel geometry. The importers build `Float64` coordinates, so a dual number
cannot be pushed in through them; deforming afterwards sidesteps that and gives a
genuinely dual-valued mesh built from a file.

In [12]:
import ForwardDiff
import FiniteDifferences

basis = beam_deformation_basis(mesh)
volume(amplitude) = mesh_signed_volume(deform_mesh(mesh, basis, amplitude))
forward = ForwardDiff.derivative(volume, 0.0)
central = FiniteDifferences.central_fdm(5, 1)(volume, 0.0)
@printf("d(volume)/d(beam amplitude): forward mode %.8g, central difference %.8g\n",
    forward, central)
@printf("relative difference %.2e\n", abs(forward / central - 1))

d(volume)/d(beam amplitude): forward mode 0.0094632313, central difference 0.0094632313
relative difference 9.28e-14


## What this notebook does not claim

Being explicit, because several of these were found the hard way:

- The surface boundary layer **does not converge** on this hull. Its residual
  plateaus. The friction coefficient and the crossflow are nonetheless stable
  under refinement, which is what the loads use.
- The domain is terminated where the thin-layer assumption fails, so nothing is
  predicted about the flow over the aftmost part of the hull. That region needs
  either higher-order boundary layer theory or a different method entirely, and
  the loads there come from the superposed vortex model instead.
- The curvature criterion estimates $\kappa$ from the turn of the normal between
  neighbouring panel centres, so it is itself mesh-sensitive. The retained
  fraction is not monotone in refinement.
- $Y_v$ over-predicts by about half. The direction is understood; it has not been
  tuned out.
- The bilge vortices are superposed, following Tanaka's first-order independence
  argument. They are not solved for, and no vortex sheet is tracked.